# 09 — Course Jurisdiction, Racing Authority and Betting-Market Context

## Purpose

The completed distance, carried-weight and starting-price studies established that the source converts international racing information into superficially consistent British-style text without necessarily preserving its original jurisdictional meaning.

A parsed value may therefore be arithmetically valid while remaining contextually ambiguous:

- a source distance expressed in miles and furlongs may approximate an official metric distance;
- a source weight expressed in stones and pounds may represent a converted official kilogram declaration;
- a fractional value in `sp` may represent a conventional fixed-odds starting price, a fractionalised tote or pari-mutuel return, or another published market measure.

These values must not be compared across jurisdictions under false equivalence.

## Bounded question

> How can each source course or race be assigned a defensible jurisdiction, racing-authority and betting-market context, with explicit evidence and confidence, so that international distance, carried-weight and source price-or-return values are not compared under false equivalence?

## Starting point

Notebook 04 already established a bounded candidate course-mapping method:

- preserve the exact raw `course` value;
- derive candidate jurisdiction from recognised terminal codes, historical suffix links, curated British configurations and bounded collision rules;
- remove only recognised terminal jurisdiction suffixes;
- retain meaningful venue or configuration markers such as `(AW)`, `(July)`, `(RH)` and `(Perth)`;
- combine candidate jurisdiction and candidate course label as the provisional venue/configuration identity.

That work assigned all **189,043** provisional races to a candidate jurisdiction and reduced **528** raw course values to **395** jurisdiction-qualified candidate venue/configuration identities. It remains candidate mapping rather than independently verified racing-authority or betting-market classification.

This notebook will reuse that implementation rather than rebuild course parsing.

## Intended outputs

Where evidence permits, the study will record:

- canonical course and jurisdiction identity;
- raw course-name variants and relevant date periods;
- governing racing authority;
- native distance convention;
- native carried-weight convention;
- principal betting or wagering system;
- the likely meaning of source `sp`;
- whether published prices or returns normally cover every runner, the winner only, or selected finishers;
- known source normalisations;
- evidence source and retrieval details;
- classification confidence;
- unresolved, mixed or period-dependent cases.

The study will test whether each conclusion is supportable at jurisdiction, racing-authority, course, date-period or individual-race level.

## Boundaries

This notebook will not:

- overwrite Notebook 04 candidate mappings;
- treat a jurisdiction suffix as proof of betting-market semantics;
- force one classification where evidence is mixed or changes over time;
- reopen the completed arithmetic parsers;
- attempt to document every racing rule worldwide;
- begin final target-schema design.

Raw values, candidate mappings, external evidence and interpretive classifications will remain separate. Unsupported cases will remain explicitly unresolved.

In [1]:
from pathlib import Path
import sqlite3

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

DATA_ROW_PREDICATE = "rowid <> 1"

assert DB_PATH.exists(), f"Source database not found: {DB_PATH}"

connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {DB_PATH}")
print("SQLite access: read-only")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
SQLite access: read-only


In [2]:
# Load one record per provisional race from the immutable source database.
# The established provisional race identity is date + course + off.
# race_name and type are retained because Notebook 04 uses them for bounded
# race-context collision rules such as Ascot and Newcastle.

import pandas as pd

provisional_races = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        MIN(race_name) AS race_name,
        MIN(type) AS type,
        COUNT(*) AS source_runner_records
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off
    ORDER BY
        date,
        course,
        off
    """,
    connection,
)

print(f"Provisional races loaded: {len(provisional_races):,}")
provisional_races.head()

Provisional races loaded: 189,043


,date,course,off,race_name,type,source_runner_records
0,2015-01-01,Aqueduct (USA),6:20,Affectionately Stakes () (4yo+ Fillies & Mares...,Flat,8
1,2015-01-01,Ascot (AUS),6:50,La Trice Classic (Fillies & Mares),Flat,8
2,2015-01-01,Ascot (AUS),8:50,Golden River Development Perth Cup (Handicap),Flat,16
3,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,Hurdle,10
4,2015-01-01,Catterick,1:05,Buy Your 2015 Annual Badge Today Handicap Hurdle,Hurdle,11


In [3]:
# Confirm that the loaded frame contains exactly one row for every established
# provisional race identity: date + course + off.

duplicate_race_keys = provisional_races.duplicated(
    subset=["date", "course", "off"],
    keep=False,
)

print(f"Provisional races: {len(provisional_races):,}")
print(f"Rows in duplicated provisional race keys: {duplicate_race_keys.sum():,}")

assert len(provisional_races) == 189_043
assert duplicate_race_keys.sum() == 0

Provisional races: 189,043
Rows in duplicated provisional race keys: 0


In [4]:
# Extract the settled jurisdiction-mapping definitions from Notebook 04 into
# a reusable module without manually retyping its curated mappings.
#
# This reads Notebook 04, selects the established mapping constants and helper
# functions by name, and writes their original source into:
# src/inside_rails/course_jurisdiction.py

import ast
import json
from pathlib import Path

NOTEBOOK_04_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "04_course_jurisdiction_and_surface_mapping.ipynb"
)

MODULE_PATH = (
    PROJECT_ROOT
    / "src"
    / "inside_rails"
    / "course_jurisdiction.py"
)

TARGET_NAMES = {
    "terminal_code_to_jurisdiction",
    "historical_course_to_code",
    "curated_british_course_configurations",
    "established_unsuffixed_british_courses",
    "extract_terminal_jurisdiction_code",
    "derive_candidate_race_jurisdiction",
}

with NOTEBOOK_04_PATH.open("r", encoding="utf-8") as notebook_file:
    notebook_04 = json.load(notebook_file)

selected_blocks = []

for cell in notebook_04["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    lines = source.splitlines(keepends=True)

    for node in tree.body:
        defined_names = set()

        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            defined_names.add(node.name)

        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]

            for target in targets:
                if isinstance(target, ast.Name):
                    defined_names.add(target.id)

        if defined_names & TARGET_NAMES:
            selected_blocks.append(
                "".join(lines[node.lineno - 1 : node.end_lineno])
            )

found_names = set()

for block in selected_blocks:
    block_tree = ast.parse(block)

    for node in block_tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            found_names.add(node.name)
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if isinstance(target, ast.Name):
                    found_names.add(target.id)

missing_names = TARGET_NAMES - found_names

assert not missing_names, (
    "Notebook 04 definitions were not all found: "
    f"{sorted(missing_names)}"
)

module_source = '''"""Reusable course and candidate-jurisdiction mapping rules.

Extracted from the settled implementation validated in Notebook 04.
Raw source values remain unchanged; these functions produce candidate
canonicalisation attributes and retain mapping evidence.
"""

from __future__ import annotations

import re

import pandas as pd


'''

module_source += "\n\n\n".join(selected_blocks)
module_source += "\n"

MODULE_PATH.write_text(module_source, encoding="utf-8")

print(f"Created: {MODULE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Definitions extracted: {sorted(found_names)}")

Created: src/inside_rails/course_jurisdiction.py
Definitions extracted: ['curated_british_course_configurations', 'derive_candidate_race_jurisdiction', 'established_unsuffixed_british_courses', 'extract_terminal_jurisdiction_code', 'historical_course_to_code', 'terminal_code_to_jurisdiction']


In [5]:
# Rebuild the module with self-contained literal mappings.
#
# Notebook 04 is executed only here as a one-off extraction source. The
# resulting production module will not depend on Notebook 04 variables,
# DataFrames or execution state.

import ast
import contextlib
import io
import json
import sys

with NOTEBOOK_04_PATH.open("r", encoding="utf-8") as notebook_file:
    notebook_04 = json.load(notebook_file)

extraction_namespace = {}

# Execute Notebook 04 cells only until the final jurisdiction function has
# been defined, suppressing its existing analytical output.
with contextlib.redirect_stdout(io.StringIO()):
    for cell in notebook_04["cells"]:
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        exec(compile(source, str(NOTEBOOK_04_PATH), "exec"), extraction_namespace)

        if "derive_candidate_race_jurisdiction" in extraction_namespace:
            break

literal_names = [
    "terminal_code_to_jurisdiction",
    "historical_course_to_code",
    "curated_british_course_configurations",
    "established_unsuffixed_british_courses",
]

function_names = [
    "extract_terminal_jurisdiction_code",
    "derive_candidate_race_jurisdiction",
]

function_blocks = {}

for cell in notebook_04["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    lines = source.splitlines(keepends=True)

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in function_names:
            function_blocks[node.name] = "".join(
                lines[node.lineno - 1 : node.end_lineno]
            )

assert set(function_blocks) == set(function_names)

module_parts = [
    '''"""Reusable course and candidate-jurisdiction mapping rules.

Extracted from the settled implementation validated in Notebook 04.
Raw source values remain unchanged; derived values remain candidate mappings
with explicit evidence labels.
"""

from __future__ import annotations

import re

import pandas as pd
'''
]

for name in literal_names:
    value = extraction_namespace[name]
    module_parts.append(f"{name} = {value!r}\n")

for name in function_names:
    module_parts.append(function_blocks[name])

MODULE_PATH.write_text(
    "\n\n".join(module_parts) + "\n",
    encoding="utf-8",
)

# Remove the failed partial import so the corrected file can be loaded next.
sys.modules.pop("inside_rails.course_jurisdiction", None)

print(f"Rebuilt: {MODULE_PATH.relative_to(PROJECT_ROOT)}")
print(
    "Historical course links: "
    f"{len(extraction_namespace['historical_course_to_code']):,}"
)
print(
    "Established unsuffixed British courses: "
    f"{len(extraction_namespace['established_unsuffixed_british_courses']):,}"
)

,measure,value
0,non-colliding comparison labels,133
1,later forms first seen from 2025-10-15 onward,133
2,later forms first seen during October 2025,47
3,pairs with no temporal overlap,133
4,pairs with a gap of 31 days or fewer,37


,measure,value
0,all unsuffixed raw course values,189
1,historically linked unsuffixed values,130
2,remaining unlinked unsuffixed values,59
3,runner rows in remaining values,702255
4,provisional races in remaining values,80795


,date,course,off,race_name,race_id,ran
0,2023-06-26,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,842254,6
1,2025-11-15,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),908281,11
2,2025-11-15,Newcastle,05:45,The Newcastle Herald Hunter (Handicap) (3yo+) ...,908282,16
3,2026-03-06,Newcastle,05:55,Horsepower Newcastle Stakes (Handicap) (Turf),914933,10


,type,provisional_races,distinct_raw_courses,distinct_race_names,first_date,last_date
0,NH Flat,3019,53,2492,2015-01-01,2026-05-26
1,Flat,14,4,7,2016-04-26,2025-04-29
2,Chase,2,2,2,2022-04-03,2025-11-11
3,Hurdle,1,1,1,2024-02-24,2024-02-24


,type,provisional_races,distinct_raw_courses,distinct_race_names,first_date,last_date
0,NH Flat,4421,92,3287,2015-01-01,2026-05-27
1,Flat,15,5,8,2016-04-26,2025-04-29
2,Chase,1,1,1,2025-11-11,2025-11-11


,course,explicit_turf_marker,provisional_races
0,Ascot,False,28
1,Ascot,True,21
2,Newcastle,True,3
3,Sandown,False,7


Rebuilt: src/inside_rails/course_jurisdiction.py
Historical course links: 133
Established unsuffixed British courses: 58


In [6]:
# Import the rebuilt reusable jurisdiction module.
# Removing any cached failed import ensures Python reads the corrected file.

import sys
import importlib

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

sys.modules.pop("inside_rails.course_jurisdiction", None)

course_jurisdiction = importlib.import_module(
    "inside_rails.course_jurisdiction"
)

print("Module imported successfully.")
print(f"Module file: {course_jurisdiction.__file__}")

Module imported successfully.
Module file: /home/rob/Documents/inside-rails-horse-racing/src/inside_rails/course_jurisdiction.py


In [7]:
# Add the two settled Notebook 04 helper constants that the reusable module
# requires, then reload the module before applying the mapping.

module_source = MODULE_PATH.read_text(encoding="utf-8")

helper_source = f"""
terminal_code_pattern = {extraction_namespace["terminal_code_pattern"]!r}

recognised_terminal_jurisdiction_codes = {
    extraction_namespace["recognised_terminal_jurisdiction_codes"]!r
}
"""

module_source = module_source.replace(
    "import pandas as pd\n",
    "import pandas as pd\n" + helper_source,
    1,
)

MODULE_PATH.write_text(module_source, encoding="utf-8")

sys.modules.pop("inside_rails.course_jurisdiction", None)

course_jurisdiction = importlib.import_module(
    "inside_rails.course_jurisdiction"
)

candidate_jurisdiction_evidence = provisional_races.copy()

candidate_jurisdiction_evidence[
    ["candidate_jurisdiction", "jurisdiction_evidence_source"]
] = candidate_jurisdiction_evidence.apply(
    course_jurisdiction.derive_candidate_race_jurisdiction,
    axis=1,
)

unresolved_races = (
    candidate_jurisdiction_evidence["candidate_jurisdiction"] == "unresolved"
).sum()

print(
    "Provisional races assigned a candidate jurisdiction: "
    f"{len(candidate_jurisdiction_evidence) - unresolved_races:,}"
)
print(f"Provisional races unresolved: {unresolved_races:,}")

Provisional races assigned a candidate jurisdiction: 189,043
Provisional races unresolved: 0


In [8]:
# Add the settled Notebook 04 candidate course-label helper to the reusable
# module, reload it, and validate the established identity count.
#
# The rule removes only a recognised terminal jurisdiction suffix.
# Meaningful configuration markers such as (AW), (July), (RH) and (Perth)
# remain part of the candidate course label.

module_source = MODULE_PATH.read_text(encoding="utf-8")

course_label_helper = '''
def derive_candidate_course_label(course_name):
    """Remove only a recognised terminal jurisdiction suffix."""
    course_text = str(course_name)
    terminal_code = extract_terminal_jurisdiction_code(course_text)

    if terminal_code is None:
        return course_text

    return terminal_code_pattern.sub("", course_text).rstrip()
'''

if "def derive_candidate_course_label" not in module_source:
    module_source += "\n\n" + course_label_helper

MODULE_PATH.write_text(module_source, encoding="utf-8")

sys.modules.pop("inside_rails.course_jurisdiction", None)

course_jurisdiction = importlib.import_module(
    "inside_rails.course_jurisdiction"
)

candidate_jurisdiction_evidence["candidate_course_label"] = (
    candidate_jurisdiction_evidence["course"].map(
        course_jurisdiction.derive_candidate_course_label
    )
)

candidate_venue_count = (
    candidate_jurisdiction_evidence[
        ["candidate_jurisdiction", "candidate_course_label"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Candidate venue/configuration identities: "
    f"{candidate_venue_count:,}"
)

Candidate venue/configuration identities: 395


In [9]:
# Validate the remaining settled Notebook 04 identity findings:
# - 135 candidate identities represented by multiple raw course forms;
# - no same-date collisions between those raw forms.

candidate_identity_forms = (
    candidate_jurisdiction_evidence
    .groupby(
        ["candidate_jurisdiction", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        distinct_raw_course_forms=("course", "nunique"),
    )
)

multi_form_identity_count = (
    candidate_identity_forms["distinct_raw_course_forms"] > 1
).sum()

same_date_collisions = (
    candidate_jurisdiction_evidence
    .groupby(
        [
            "date",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        as_index=False,
    )
    .agg(
        distinct_raw_course_forms=("course", "nunique"),
    )
)

same_date_collision_count = (
    same_date_collisions["distinct_raw_course_forms"] > 1
).sum()

print(
    "Candidate identities with multiple raw forms: "
    f"{multi_form_identity_count:,}"
)
print(
    "Same-date candidate identity collisions: "
    f"{same_date_collision_count:,}"
)

assert multi_form_identity_count == 135
assert same_date_collision_count == 0

Candidate identities with multiple raw forms: 135
Same-date candidate identity collisions: 0


In [10]:
# Lock the reusable module to the complete set of settled Notebook 04
# jurisdiction and candidate-identity findings.

assigned_candidate_jurisdictions = (
    candidate_jurisdiction_evidence["candidate_jurisdiction"] != "unresolved"
).sum()

assert assigned_candidate_jurisdictions == 189_043
assert candidate_venue_count == 395
assert multi_form_identity_count == 135
assert same_date_collision_count == 0

print("Notebook 04 reusable mapping regression checks passed.")

Notebook 04 reusable mapping regression checks passed.


In [11]:
# Write an independent validation script for the reusable course-jurisdiction
# module. The script opens the immutable source database read-only and confirms
# that future changes still reproduce Notebook 04's settled mapping counts.

VALIDATION_SCRIPT_PATH = (
    PROJECT_ROOT
    / "scripts"
    / "validate_course_jurisdiction.py"
)

validation_script = '''"""Validate the reusable course and jurisdiction mapping.

Usage:
    PYTHONPATH=src python scripts/validate_course_jurisdiction.py \\
        data/raw/form_2015-present/form_2015-present/raceform.db
"""

from __future__ import annotations

import argparse
import sqlite3
from pathlib import Path

import pandas as pd

from inside_rails.course_jurisdiction import (
    derive_candidate_course_label,
    derive_candidate_race_jurisdiction,
)


DATA_ROW_PREDICATE = "rowid <> 1"


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("database", type=Path)
    args = parser.parse_args()

    database_path = args.database.resolve()

    if not database_path.exists():
        raise FileNotFoundError(database_path)

    connection = sqlite3.connect(
        f"file:{database_path}?mode=ro",
        uri=True,
    )

    try:
        provisional_races = pd.read_sql_query(
            f"""
            SELECT
                date,
                course,
                off,
                MIN(race_name) AS race_name,
                MIN(type) AS type
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off
            """,
            connection,
        )
    finally:
        connection.close()

    duplicate_keys = provisional_races.duplicated(
        subset=["date", "course", "off"],
        keep=False,
    ).sum()

    mapped = provisional_races.copy()

    mapped[
        ["candidate_jurisdiction", "jurisdiction_evidence_source"]
    ] = mapped.apply(
        derive_candidate_race_jurisdiction,
        axis=1,
    )

    mapped["candidate_course_label"] = mapped["course"].map(
        derive_candidate_course_label
    )

    assigned_races = (
        mapped["candidate_jurisdiction"] != "unresolved"
    ).sum()

    candidate_venue_count = (
        mapped[
            ["candidate_jurisdiction", "candidate_course_label"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    identity_forms = (
        mapped
        .groupby(
            ["candidate_jurisdiction", "candidate_course_label"],
            as_index=False,
        )
        .agg(distinct_raw_course_forms=("course", "nunique"))
    )

    multi_form_identity_count = (
        identity_forms["distinct_raw_course_forms"] > 1
    ).sum()

    date_identity_forms = (
        mapped
        .groupby(
            [
                "date",
                "candidate_jurisdiction",
                "candidate_course_label",
            ],
            as_index=False,
        )
        .agg(distinct_raw_course_forms=("course", "nunique"))
    )

    same_date_collision_count = (
        date_identity_forms["distinct_raw_course_forms"] > 1
    ).sum()

    assert len(provisional_races) == 189_043
    assert duplicate_keys == 0
    assert assigned_races == 189_043
    assert candidate_venue_count == 395
    assert multi_form_identity_count == 135
    assert same_date_collision_count == 0

    print("Course-jurisdiction validation passed.")
    print(f"Provisional races: {len(provisional_races):,}")
    print(f"Candidate jurisdictions assigned: {assigned_races:,}")
    print(f"Candidate venue/configuration identities: {candidate_venue_count:,}")
    print(f"Multi-form candidate identities: {multi_form_identity_count:,}")
    print(f"Same-date candidate identity collisions: {same_date_collision_count:,}")


if __name__ == "__main__":
    main()
'''

VALIDATION_SCRIPT_PATH.write_text(
    validation_script,
    encoding="utf-8",
)

print(
    f"Created: {VALIDATION_SCRIPT_PATH.relative_to(PROJECT_ROOT)}"
)

Created: scripts/validate_course_jurisdiction.py


In [12]:
# Run the independent validator against the immutable source database.

import subprocess

result = subprocess.run(
    [
        sys.executable,
        str(VALIDATION_SCRIPT_PATH),
        str(DB_PATH),
    ],
    cwd=PROJECT_ROOT,
    env={
        **dict(__import__("os").environ),
        "PYTHONPATH": str(PROJECT_ROOT / "src"),
    },
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

assert result.returncode == 0

Course-jurisdiction validation passed.
Provisional races: 189,043
Candidate jurisdictions assigned: 189,043
Candidate venue/configuration identities: 395
Multi-form candidate identities: 135
Same-date candidate identity collisions: 0



## Reusable course and jurisdiction mapping

Notebook 04's settled candidate course and jurisdiction logic has now been extracted into `src/inside_rails/course_jurisdiction.py`.

An independent validation script has also been added at `scripts/validate_course_jurisdiction.py`.

The reusable implementation reproduces the established Notebook 04 results:

- 189,043 provisional races
- 189,043 candidate-jurisdiction assignments
- 395 candidate venue/configuration identities
- 135 candidate identities represented by multiple raw course forms
- zero same-date candidate-identity collisions

Notebook 09 will use this validated mapping as its starting population. The mapping remains candidate canonicalisation with retained evidence labels; it is not yet a complete racing-authority or betting-market classification.

In [13]:
# Summarise the validated starting population by candidate jurisdiction.
# This establishes the scale of each jurisdiction before we begin adding
# racing-authority, distance, weight and betting-market context.

jurisdiction_population = (
    candidate_jurisdiction_evidence
    .groupby("candidate_jurisdiction", as_index=False)
    .agg(
        provisional_races=("course", "size"),
        distinct_raw_courses=("course", "nunique"),
        candidate_course_identities=("candidate_course_label", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["provisional_races", "candidate_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

jurisdiction_population["share_of_all_races"] = (
    jurisdiction_population["provisional_races"]
    / len(candidate_jurisdiction_evidence)
)

jurisdiction_population

,candidate_jurisdiction,provisional_races,distinct_raw_courses,candidate_course_identities,first_date,last_date,share_of_all_races
0,Great Britain,111634,65,65,2015-01-01,2026-05-27,0.590522
1,Ireland,30783,50,27,2015-01-01,2026-05-27,0.162836
2,France,20533,92,73,2015-01-03,2026-05-26,0.108616
3,Hong Kong,7481,4,2,2015-01-25,2026-05-27,0.039573
4,United States,6105,77,56,2015-01-01,2026-05-26,0.032294
5,Australia,4059,74,51,2015-01-01,2026-05-23,0.021471
6,United Arab Emirates,2451,9,5,2015-01-04,2026-04-11,0.012965
7,Japan,1559,30,21,2015-01-04,2026-05-24,0.008247
8,Germany,731,25,17,2015-04-05,2026-05-25,0.003867
9,Canada,471,5,4,2015-04-18,2026-05-02,0.002491


## Jurisdiction population and study order

The validated mapping assigns the 189,043 provisional races across 36 candidate jurisdictions.

Coverage is highly concentrated:

- Great Britain: 111,634 races (59.05%)
- Ireland: 30,783 races (16.28%)
- France: 20,533 races (10.86%)
- Hong Kong: 7,481 races (3.96%)
- United States: 6,105 races (3.23%)
- Australia: 4,059 races (2.15%)

Great Britain, Ireland and France together account for approximately 86.2% of the current source.

The authority and betting-market investigation will therefore proceed in evidence-led groups:

1. Great Britain and Ireland, where the source presentation is closest to its apparent native format
2. France, where Notebook 08 already demonstrated mixed price-and-return semantics
3. Hong Kong, the United States, Australia, the United Arab Emirates and Japan
4. smaller jurisdictions, grouped only where a shared racing authority or wagering framework is genuinely supported

This ordering is for investigation efficiency only. It does not imply that one jurisdiction-level classification will necessarily apply to every course, date period or race.

In [14]:
# Inspect the Great Britain and Ireland candidate course identities that will
# form the first authority and betting-market evidence group.

british_irish_course_inventory = (
    candidate_jurisdiction_evidence.loc[
        candidate_jurisdiction_evidence["candidate_jurisdiction"].isin(
            ["Great Britain", "Ireland"]
        )
    ]
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("course", "size"),
        distinct_raw_course_forms=("course", "nunique"),
        raw_course_forms=(
            "course",
            lambda values: " | ".join(sorted(set(values))),
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "candidate_jurisdiction",
            "provisional_races",
            "candidate_course_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Great Britain candidate identities: "
    f"{(british_irish_course_inventory['candidate_jurisdiction'] == 'Great Britain').sum():,}"
)
print(
    "Ireland candidate identities: "
    f"{(british_irish_course_inventory['candidate_jurisdiction'] == 'Ireland').sum():,}"
)

british_irish_course_inventory

Great Britain candidate identities: 65
Ireland candidate identities: 27


,candidate_jurisdiction,candidate_course_label,provisional_races,distinct_raw_course_forms,raw_course_forms,first_date,last_date
0,Great Britain,Wolverhampton (AW),7042,1,Wolverhampton (AW),2015-01-02,2026-05-18
1,Great Britain,Kempton (AW),5046,1,Kempton (AW),2015-01-07,2026-05-27
2,Great Britain,Lingfield (AW),4803,1,Lingfield (AW),2015-01-03,2026-05-12
3,Great Britain,Newcastle (AW),4409,1,Newcastle (AW),2016-05-17,2026-05-19
4,Great Britain,Chelmsford (AW),4310,1,Chelmsford (AW),2015-01-11,2026-03-26
5,Great Britain,Southwell (AW),3816,1,Southwell (AW),2015-01-01,2026-05-21
6,Great Britain,Doncaster,2805,1,Doncaster,2015-01-09,2026-05-16
7,Great Britain,Ayr,2325,1,Ayr,2015-01-02,2026-05-20
8,Great Britain,Chepstow,2318,1,Chepstow,2015-01-30,2026-05-21
9,Great Britain,Haydock,2269,1,Haydock,2015-01-17,2026-05-23


In [15]:
# Summarise Great Britain and Ireland by source race type.
# This tests whether a single jurisdiction-level authority classification may
# need separate Flat and National Hunt contexts.

british_irish_type_summary = (
    candidate_jurisdiction_evidence.loc[
        candidate_jurisdiction_evidence["candidate_jurisdiction"].isin(
            ["Great Britain", "Ireland"]
        )
    ]
    .groupby(
        ["candidate_jurisdiction", "type"],
        as_index=False,
    )
    .agg(
        provisional_races=("course", "size"),
        candidate_course_identities=("candidate_course_label", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["candidate_jurisdiction", "provisional_races"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

british_irish_type_summary

,candidate_jurisdiction,type,provisional_races,candidate_course_identities,first_date,last_date
0,Great Britain,Flat,70218,43,2015-01-01,2026-05-27
1,Great Britain,Hurdle,22645,43,2015-01-01,2026-05-27
2,Great Britain,Chase,15671,43,2015-01-01,2026-05-27
3,Great Britain,NH Flat,3100,44,2015-01-01,2026-05-26
4,Ireland,Flat,14763,26,2015-01-01,2026-05-26
5,Ireland,Hurdle,9667,24,2015-01-01,2026-05-27
6,Ireland,Chase,4833,23,2015-01-01,2026-05-25
7,Ireland,NH Flat,1520,24,2015-01-03,2026-05-27


## Classification grain for rules and regulatory context

The eventual database should not assign a single timeless `racing_authority` value to each jurisdiction or course.

The useful classification grain is:

- jurisdiction
- regulatory authority
- authority role
- racing code
- applicable rules framework
- effective date period
- betting or wagering regime
- course-level or race-level exception
- evidence source
- confidence

This distinction is necessary because one jurisdiction may contain:

- one authority covering several racing codes
- separate regulatory and administrative bodies
- rules that differ between Flat, Hurdle, Chase and NH Flat racing
- historical changes in regulator, rules or wagering arrangements
- exceptions that apply only to a course, meeting, race type or date period

Notebook 09 will therefore investigate authority and rules context separately.

The source `type` field will be used as evidence for racing-code segmentation, but it will not by itself determine the governing authority or complete applicable rules framework.

Final relational schema design remains deferred until these relationships and their historical boundaries are supported by evidence.

In [16]:
# Build the first-pass rules-context investigation units.
#
# This does not yet assign a regulator or formal rules framework.
# It only identifies the observed combinations of:
# - candidate jurisdiction;
# - source racing code;
# - active date period;
# - race and course coverage.
#
# These combinations will guide the external authority and rules research.

rules_context_units = (
    candidate_jurisdiction_evidence
    .groupby(
        [
            "candidate_jurisdiction",
            "type",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("course", "size"),
        candidate_course_identities=("candidate_course_label", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "type",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print(f"Observed jurisdiction-and-code units: {len(rules_context_units):,}")

rules_context_units

Observed jurisdiction-and-code units: 56


,candidate_jurisdiction,type,provisional_races,candidate_course_identities,first_date,last_date
0,Great Britain,Flat,70218,43,2015-01-01,2026-05-27
1,Great Britain,Hurdle,22645,43,2015-01-01,2026-05-27
2,Great Britain,Chase,15671,43,2015-01-01,2026-05-27
3,France,Flat,15514,65,2015-01-03,2026-05-24
4,Ireland,Flat,14763,26,2015-01-01,2026-05-26
5,Ireland,Hurdle,9667,24,2015-01-01,2026-05-27
6,Hong Kong,Flat,7481,2,2015-01-25,2026-05-27
7,United States,Flat,6023,50,2015-01-01,2026-05-26
8,Ireland,Chase,4833,23,2015-01-01,2026-05-25
9,Australia,Flat,4059,51,2015-01-01,2026-05-23


## Observed jurisdiction-and-code units

The source contains 56 observed combinations of candidate jurisdiction and source `type`.

These combinations are investigation units, not yet formal rules-framework assignments.

The large populations support immediate separate study of:

- Great Britain: Flat, Hurdle, Chase and NH Flat
- Ireland: Flat, Hurdle, Chase and NH Flat
- France: Flat, Hurdle and Chase
- United States: predominantly Flat, with smaller Hurdle and Chase populations
- Japan: predominantly Flat, with small obstacle-racing populations

Several very small combinations require caution. A source label such as `Hurdle`, `Chase` or `NH Flat` does not by itself prove that the race falls under a British-style rules category with the same regulatory meaning.

The next stage must therefore distinguish:

- the source's broad race-type label
- the jurisdiction's native racing code
- the applicable formal rules framework
- the responsible regulator
- the effective date period

Thin combinations will remain unresolved until their race context and governing rules can be verified.

In [17]:
# Record the first evidence-backed Great Britain rules-context assignments.
#
# These rows assign the regulatory authority and broad native racing code.
# They do not yet claim that one unchanged edition of the Rules of Racing
# applied throughout 2015–2026. Formal rules-version periods remain unresolved.

great_britain_rules_context = pd.DataFrame(
    [
        {
            "candidate_jurisdiction": "Great Britain",
            "source_type": "Flat",
            "regulatory_authority": "British Horseracing Authority",
            "authority_abbreviation": "BHA",
            "native_racing_code": "Flat racing",
            "rules_framework": "BHA Rules of Racing",
            "rules_version_period": "requires date-period segmentation",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
            "evidence_url": (
                "https://www.britishhorseracing.com/"
                "regulation/rules-guides/"
            ),
        },
        {
            "candidate_jurisdiction": "Great Britain",
            "source_type": "Hurdle",
            "regulatory_authority": "British Horseracing Authority",
            "authority_abbreviation": "BHA",
            "native_racing_code": "Hurdle racing",
            "rules_framework": "BHA Rules of Racing",
            "rules_version_period": "requires date-period segmentation",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
            "evidence_url": (
                "https://www.britishhorseracing.com/"
                "regulation/rules-guides/"
            ),
        },
        {
            "candidate_jurisdiction": "Great Britain",
            "source_type": "Chase",
            "regulatory_authority": "British Horseracing Authority",
            "authority_abbreviation": "BHA",
            "native_racing_code": "Steeple Chase racing",
            "rules_framework": "BHA Rules of Racing",
            "rules_version_period": "requires date-period segmentation",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
            "evidence_url": (
                "https://www.britishhorseracing.com/"
                "regulation/rules-guides/"
            ),
        },
        {
            "candidate_jurisdiction": "Great Britain",
            "source_type": "NH Flat",
            "regulatory_authority": "British Horseracing Authority",
            "authority_abbreviation": "BHA",
            "native_racing_code": "National Hunt Flat racing",
            "rules_framework": "BHA Rules of Racing",
            "rules_version_period": "requires date-period segmentation",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
            "evidence_url": (
                "https://www.britishhorseracing.com/"
                "regulation/glossary-of-race-types/"
            ),
        },
    ]
)

great_britain_rules_context

,candidate_jurisdiction,source_type,regulatory_authority,authority_abbreviation,native_racing_code,rules_framework,rules_version_period,classification_level,evidence_status,confidence,evidence_url
0,Great Britain,Flat,British Horseracing Authority,BHA,Flat racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
1,Great Britain,Hurdle,British Horseracing Authority,BHA,Hurdle racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
2,Great Britain,Chase,British Horseracing Authority,BHA,Steeple Chase racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
3,Great Britain,NH Flat,British Horseracing Authority,BHA,National Hunt Flat racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/glossary-of-race-types/


In [18]:
# Join the evidence-backed Great Britain rules context to the observed
# jurisdiction-and-code units and confirm that every British source type
# receives exactly one rules-context assignment.

great_britain_rules_coverage = (
    rules_context_units.loc[
        rules_context_units["candidate_jurisdiction"] == "Great Britain"
    ]
    .merge(
        great_britain_rules_context,
        left_on=["candidate_jurisdiction", "type"],
        right_on=["candidate_jurisdiction", "source_type"],
        how="left",
        validate="one_to_one",
    )
)

missing_assignments = (
    great_britain_rules_coverage["regulatory_authority"].isna().sum()
)

print(
    "Great Britain jurisdiction-and-code units: "
    f"{len(great_britain_rules_coverage):,}"
)
print(f"Missing rules-context assignments: {missing_assignments:,}")

assert len(great_britain_rules_coverage) == 4
assert missing_assignments == 0

great_britain_rules_coverage

Great Britain jurisdiction-and-code units: 4
Missing rules-context assignments: 0


,candidate_jurisdiction,type,provisional_races,candidate_course_identities,first_date,last_date,source_type,regulatory_authority,authority_abbreviation,native_racing_code,rules_framework,rules_version_period,classification_level,evidence_status,confidence,evidence_url
0,Great Britain,Flat,70218,43,2015-01-01,2026-05-27,Flat,British Horseracing Authority,BHA,Flat racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
1,Great Britain,Hurdle,22645,43,2015-01-01,2026-05-27,Hurdle,British Horseracing Authority,BHA,Hurdle racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
2,Great Britain,Chase,15671,43,2015-01-01,2026-05-27,Chase,British Horseracing Authority,BHA,Steeple Chase racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/rules-guides/
3,Great Britain,NH Flat,3100,44,2015-01-01,2026-05-26,NH Flat,British Horseracing Authority,BHA,National Hunt Flat racing,BHA Rules of Racing,requires date-period segmentation,jurisdiction_and_racing_code,official_authority_evidence,high,https://www.britishhorseracing.com/regulation/glossary-of-race-types/


In [19]:
# Record Ireland's first-pass rules contexts with historical segmentation.
#
# The available official evidence establishes:
# - 2015–2017 source races pre-date the IHRB and fall within the former
#   Turf Club / Irish National Hunt Steeplechase Committee regime;
# - the IHRB was established in 2018 and now makes and enforces the rules;
# - Flat racing uses the Rules of Racing;
# - Hurdle, Chase and Irish National Hunt Flat racing require the
#   Irish National Hunt Steeplechase rules context;
# - Horse Racing Ireland remains a separate industry-administration body.
#
# The evidence currently establishes the transition year, not an exact
# effective day, so the period boundary remains year-level.

ireland_rules_context = pd.DataFrame(
    [
        {
            "candidate_jurisdiction": "Ireland",
            "source_type": source_type,
            "period_start_year": 2015,
            "period_end_year": 2017,
            "regulatory_authority": (
                "Turf Club"
                if source_type == "Flat"
                else "Irish National Hunt Steeplechase Committee"
            ),
            "authority_abbreviation": (
                "Turf Club"
                if source_type == "Flat"
                else "INHSC"
            ),
            "administrative_body": "Horse Racing Ireland",
            "native_racing_code": (
                "Flat racing"
                if source_type == "Flat"
                else {
                    "Hurdle": "Hurdle racing",
                    "Chase": "Steeplechase racing",
                    "NH Flat": "Irish National Hunt Flat racing",
                }[source_type]
            ),
            "rules_framework": (
                "Irish Rules of Racing"
                if source_type == "Flat"
                else "Irish National Hunt Steeplechase Rules"
            ),
            "classification_level": (
                "jurisdiction_racing_code_and_year_period"
            ),
            "evidence_status": "official_historical_authority_evidence",
            "confidence": "high",
        }
        for source_type in ["Flat", "Hurdle", "Chase", "NH Flat"]
    ]
    + [
        {
            "candidate_jurisdiction": "Ireland",
            "source_type": source_type,
            "period_start_year": 2018,
            "period_end_year": None,
            "regulatory_authority": (
                "Irish Horseracing Regulatory Board"
            ),
            "authority_abbreviation": "IHRB",
            "administrative_body": "Horse Racing Ireland",
            "native_racing_code": (
                "Flat racing"
                if source_type == "Flat"
                else {
                    "Hurdle": "Hurdle racing",
                    "Chase": "Steeplechase racing",
                    "NH Flat": "Irish National Hunt Flat racing",
                }[source_type]
            ),
            "rules_framework": (
                "Irish Rules of Racing"
                if source_type == "Flat"
                else "Irish National Hunt Steeplechase Rules"
            ),
            "classification_level": (
                "jurisdiction_racing_code_and_year_period"
            ),
            "evidence_status": "official_current_authority_evidence",
            "confidence": "high",
        }
        for source_type in ["Flat", "Hurdle", "Chase", "NH Flat"]
    ]
)

ireland_rules_context

,candidate_jurisdiction,source_type,period_start_year,period_end_year,regulatory_authority,authority_abbreviation,administrative_body,native_racing_code,rules_framework,classification_level,evidence_status,confidence
0,Ireland,Flat,2015,2017.0,Turf Club,Turf Club,Horse Racing Ireland,Flat racing,Irish Rules of Racing,jurisdiction_racing_code_and_year_period,official_historical_authority_evidence,high
1,Ireland,Hurdle,2015,2017.0,Irish National Hunt Steeplechase Committee,INHSC,Horse Racing Ireland,Hurdle racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_historical_authority_evidence,high
2,Ireland,Chase,2015,2017.0,Irish National Hunt Steeplechase Committee,INHSC,Horse Racing Ireland,Steeplechase racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_historical_authority_evidence,high
3,Ireland,NH Flat,2015,2017.0,Irish National Hunt Steeplechase Committee,INHSC,Horse Racing Ireland,Irish National Hunt Flat racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_historical_authority_evidence,high
4,Ireland,Flat,2018,NaN,Irish Horseracing Regulatory Board,IHRB,Horse Racing Ireland,Flat racing,Irish Rules of Racing,jurisdiction_racing_code_and_year_period,official_current_authority_evidence,high
5,Ireland,Hurdle,2018,NaN,Irish Horseracing Regulatory Board,IHRB,Horse Racing Ireland,Hurdle racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_current_authority_evidence,high
6,Ireland,Chase,2018,NaN,Irish Horseracing Regulatory Board,IHRB,Horse Racing Ireland,Steeplechase racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_current_authority_evidence,high
7,Ireland,NH Flat,2018,NaN,Irish Horseracing Regulatory Board,IHRB,Horse Racing Ireland,Irish National Hunt Flat racing,Irish National Hunt Steeplechase Rules,jurisdiction_racing_code_and_year_period,official_current_authority_evidence,high


In [20]:
# Split the observed Irish race population across the two regulatory periods.
# This checks that every Ireland source-type row can be assigned to exactly one
# year-bounded rules context.

ireland_races = candidate_jurisdiction_evidence.loc[
    candidate_jurisdiction_evidence["candidate_jurisdiction"] == "Ireland"
].copy()

ireland_races["race_year"] = ireland_races["date"].str[:4].astype(int)

ireland_races["regulatory_period"] = ireland_races["race_year"].map(
    lambda year: "2015–2017" if year <= 2017 else "2018 onward"
)

ireland_period_coverage = (
    ireland_races
    .groupby(
        ["type", "regulatory_period"],
        as_index=False,
    )
    .agg(
        provisional_races=("course", "size"),
        candidate_course_identities=("candidate_course_label", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["type", "first_date"],
    )
    .reset_index(drop=True)
)

ireland_period_coverage

,type,regulatory_period,provisional_races,candidate_course_identities,first_date,last_date
0,Chase,2015–2017,1301,23,2015-01-01,2017-12-31
1,Chase,2018 onward,3532,22,2018-01-01,2026-05-25
2,Flat,2015–2017,3644,26,2015-01-01,2017-12-22
3,Flat,2018 onward,11119,26,2018-01-01,2026-05-26
4,Hurdle,2015–2017,2400,24,2015-01-01,2017-12-31
5,Hurdle,2018 onward,7267,23,2018-01-01,2026-05-27
6,NH Flat,2015–2017,360,24,2015-01-03,2017-12-31
7,NH Flat,2018 onward,1160,23,2018-01-06,2026-05-27


## Bounded scope and deferral rule

Notebook 09 will not attempt to complete a worldwide racing-authority and betting-market reference.

Its purpose is to establish a defensible classification method and test that method on a small number of materially different jurisdictions.

The detailed worked studies will cover:

- Great Britain
- Ireland
- France
- Hong Kong
- United States

These jurisdictions provide contrasting examples of:

- one regulator covering several racing codes
- separate regulatory and administrative bodies
- historical authority changes
- fixed-odds and pari-mutuel environments
- source values whose meaning may differ by jurisdiction or race context

Other jurisdictions will retain their validated candidate jurisdiction and course identities but will not receive speculative authority or market classifications in this notebook.

Their detailed rules, regulator, distance, weight and wagering context will be researched later when a country-specific or analytical study requires them.

The escalation rule is:

> Add deeper jurisdiction, code, period, course or race-level context only when the planned analysis or observed source behaviour requires it.

Notebook 09 will therefore produce a reusable classification framework and several fully evidenced examples, not a shallow global catalogue.

In [21]:
# Record the first-pass France rules context.
#
# France Galop governs and regulates French thoroughbred Flat and jump racing.
# The source types are retained separately because they represent distinct
# racing codes, even though they share the same principal authority and
# overarching Rules of Racing.
#
# The small source "NH Flat" population remains unresolved pending inspection:
# that British-style label may not correspond to a native French racing code.

france_rules_context = pd.DataFrame(
    [
        {
            "candidate_jurisdiction": "France",
            "source_type": "Flat",
            "regulatory_authority": "France Galop",
            "authority_abbreviation": "France Galop",
            "native_racing_code": "Flat racing",
            "rules_framework": "France Galop Rules of Racing",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
        },
        {
            "candidate_jurisdiction": "France",
            "source_type": "Hurdle",
            "regulatory_authority": "France Galop",
            "authority_abbreviation": "France Galop",
            "native_racing_code": "Hurdle racing",
            "rules_framework": "France Galop Rules of Racing",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
        },
        {
            "candidate_jurisdiction": "France",
            "source_type": "Chase",
            "regulatory_authority": "France Galop",
            "authority_abbreviation": "France Galop",
            "native_racing_code": "Steeplechase racing",
            "rules_framework": "France Galop Rules of Racing",
            "classification_level": "jurisdiction_and_racing_code",
            "evidence_status": "official_authority_evidence",
            "confidence": "high",
        },
        {
            "candidate_jurisdiction": "France",
            "source_type": "NH Flat",
            "regulatory_authority": "France Galop",
            "authority_abbreviation": "France Galop",
            "native_racing_code": "unresolved",
            "rules_framework": "France Galop Rules of Racing",
            "classification_level": "source_type_requires_race_context_review",
            "evidence_status": "authority_supported_code_unresolved",
            "confidence": "medium",
        },
    ]
)

france_rules_context

,candidate_jurisdiction,source_type,regulatory_authority,authority_abbreviation,native_racing_code,rules_framework,classification_level,evidence_status,confidence
0,France,Flat,France Galop,France Galop,Flat racing,France Galop Rules of Racing,jurisdiction_and_racing_code,official_authority_evidence,high
1,France,Hurdle,France Galop,France Galop,Hurdle racing,France Galop Rules of Racing,jurisdiction_and_racing_code,official_authority_evidence,high
2,France,Chase,France Galop,France Galop,Steeplechase racing,France Galop Rules of Racing,jurisdiction_and_racing_code,official_authority_evidence,high
3,France,NH Flat,France Galop,France Galop,unresolved,France Galop Rules of Racing,source_type_requires_race_context_review,authority_supported_code_unresolved,medium


In [22]:
# Inspect the 23 French races labelled "NH Flat" by the source.
# The aim is to determine whether this is a genuine native French code,
# a source normalisation, or a misclassification requiring race-level review.

france_nh_flat_races = (
    candidate_jurisdiction_evidence.loc[
        (candidate_jurisdiction_evidence["candidate_jurisdiction"] == "France")
        & (candidate_jurisdiction_evidence["type"] == "NH Flat"),
        [
            "date",
            "course",
            "candidate_course_label",
            "off",
            "race_name",
            "source_runner_records",
        ],
    ]
    .sort_values(["date", "course", "off"])
    .reset_index(drop=True)
)

print(f"French source-labelled NH Flat races: {len(france_nh_flat_races):,}")

france_nh_flat_races

French source-labelled NH Flat races: 23


,date,course,candidate_course_label,off,race_name,source_runner_records
0,2020-10-30,Saint-Cloud (FR),Saint-Cloud,12:38,Prix Chloris (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),7
1,2020-11-27,Saint-Cloud (FR),Saint-Cloud,11:20,Prix Jacques de Vienne (AQPS National Hunt Flat) (3yo) (Turf),8
2,2021-10-12,Saint-Cloud (FR),Saint-Cloud,2:35,Prix Glorieuse (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),10
3,2022-04-29,Saint-Cloud (FR),Saint-Cloud,3:35,Prix dEstruval (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),7
4,2022-09-15,Longchamp (FR),Longchamp,5:50,Prix de Craon (National Hunt Flat) (AQPS (4-5yo) (Grande Course) (Turf),9
5,2022-10-11,Saint-Cloud (FR),Saint-Cloud,2:00,Prix Glorieuse (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),9
6,2022-11-09,Saint-Cloud (FR),Saint-Cloud,12:30,Prix Chloris (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),6
7,2023-09-07,Longchamp (FR),Longchamp,6:10,Prix de Craon (National Hunt Flat) (AQPS Conditions) (4-5yo) (Grande Course) (Turf),11
8,2023-09-22,Saint-Cloud (FR),Saint-Cloud,2:25,Prix du Bourbonnais (National Hunt Flat) (AQPS (3yo) (Turf),8
9,2023-10-30,Saint-Cloud (FR),Saint-Cloud,1:25,Prix Glorieuse (National Hunt Flat) (AQPS (4-5yo Fillies & Mares) (Turf),6


## Separation of source data and researched interpretation

The staging database must not depend on completing worldwide regulatory and betting-market research before race and runner reconstruction can proceed.

Three information layers will therefore remain separate.

### Source layer

The source layer preserves values exactly as supplied, including:

- source race type
- race name
- course text
- distance
- carried weight
- starting-price text
- runner and result attributes
- physical source lineage

Source values will not be overwritten when later research suggests that a label is non-native, simplified or potentially misleading.

### Structural derivation layer

The structural layer contains reproducible derivations supported directly by the source data, including:

- staging race and runner identifiers
- candidate course identity
- candidate jurisdiction
- parsed distance and weight components
- normalised result fields
- validation and quality flags

These derivations may be completed before detailed country research.

### Research interpretation layer

The interpretation layer will contain evidence-backed contextual classifications such as:

- regulatory authority
- administrative authority
- native racing code
- applicable rules framework and effective period
- wagering system
- meaning and provenance of starting-price values
- source-to-native classification mappings
- documented exceptions

Interpretation records must retain their evidence, confidence, effective period and research status.

Where research is incomplete, the interpretation remains unassigned. The source value continues to be available unchanged.

For example, French AQPS races currently labelled `NH Flat` by the source will retain that source type. Any native French classification will be added later as a researched interpretation rather than used to overwrite the source record.

This separation allows core database reconstruction to continue while jurisdiction-specific research is completed incrementally.

In [23]:
# Record unresolved interpretation questions without changing source values.
# These issues can later be resolved by a dedicated country study and then
# linked to evidence-backed enrichment records.

interpretation_research_issues = pd.DataFrame(
    [
        {
            "candidate_jurisdiction": "France",
            "source_field": "type",
            "source_value": "NH Flat",
            "issue_scope": "23 AQPS races",
            "research_question": (
                "What native France Galop classification corresponds to "
                "source-labelled NH Flat AQPS races run without obstacles?"
            ),
            "current_status": "research_required",
            "source_value_preserved": True,
            "interpretation_applied": False,
            "priority": "country_study",
        }
    ]
)

interpretation_research_issues

,candidate_jurisdiction,source_field,source_value,issue_scope,research_question,current_status,source_value_preserved,interpretation_applied,priority
0,France,type,NH Flat,23 AQPS races,What native France Galop classification corresponds to source-labelled NH Flat AQPS races run without obstacles?,research_required,True,False,country_study


In [24]:
# Summarise the worked jurisdiction examples without treating unresolved
# research questions as completed classifications.

worked_jurisdiction_status = pd.DataFrame(
    [
        {
            "candidate_jurisdiction": "Great Britain",
            "authority_context": "completed",
            "racing_code_context": "completed",
            "historical_period_context": "requires later rules-version study",
            "open_interpretation_issues": 0,
        },
        {
            "candidate_jurisdiction": "Ireland",
            "authority_context": "completed",
            "racing_code_context": "completed",
            "historical_period_context": "2015–2017 and 2018 onward identified",
            "open_interpretation_issues": 0,
        },
        {
            "candidate_jurisdiction": "France",
            "authority_context": "completed",
            "racing_code_context": "partially completed",
            "historical_period_context": "not investigated",
            "open_interpretation_issues": len(
                interpretation_research_issues.loc[
                    interpretation_research_issues[
                        "candidate_jurisdiction"
                    ] == "France"
                ]
            ),
        },
    ]
)

worked_jurisdiction_status

,candidate_jurisdiction,authority_context,racing_code_context,historical_period_context,open_interpretation_issues
0,Great Britain,completed,completed,requires later rules-version study,0
1,Ireland,completed,completed,2015–2017 and 2018 onward identified,0
2,France,completed,partially completed,not investigated,1


In [25]:
# Translate the notebook findings into concrete database-design requirements.
# These requirements allow reconstruction work to continue without waiting
# for every jurisdiction-specific research question to be resolved.

database_context_requirements = pd.DataFrame(
    [
        {
            "area": "Source preservation",
            "requirement": "Retain source type exactly as supplied",
            "database_layer": "source",
            "required_before_reconstruction": True,
        },
        {
            "area": "Jurisdiction",
            "requirement": "Store reproducible candidate jurisdiction separately",
            "database_layer": "structural_derivation",
            "required_before_reconstruction": True,
        },
        {
            "area": "Racing code",
            "requirement": "Do not treat source type as a universal native code",
            "database_layer": "research_interpretation",
            "required_before_reconstruction": False,
        },
        {
            "area": "Regulatory context",
            "requirement": "Support authority and rules records with effective periods",
            "database_layer": "research_interpretation",
            "required_before_reconstruction": False,
        },
        {
            "area": "Betting context",
            "requirement": "Preserve raw SP and interpret its market meaning separately",
            "database_layer": "source_and_research_interpretation",
            "required_before_reconstruction": True,
        },
        {
            "area": "Research issues",
            "requirement": "Track unresolved questions without forcing classifications",
            "database_layer": "research_register",
            "required_before_reconstruction": False,
        },
    ]
)

database_context_requirements

,area,requirement,database_layer,required_before_reconstruction
0,Source preservation,Retain source type exactly as supplied,source,True
1,Jurisdiction,Store reproducible candidate jurisdiction separately,structural_derivation,True
2,Racing code,Do not treat source type as a universal native code,research_interpretation,False
3,Regulatory context,Support authority and rules records with effective periods,research_interpretation,False
4,Betting context,Preserve raw SP and interpret its market meaning separately,source_and_research_interpretation,True
5,Research issues,Track unresolved questions without forcing classifications,research_register,False


In [26]:
# Separate true reconstruction prerequisites from research that can be
# completed later through jurisdiction-specific enrichment studies.

reconstruction_readiness = (
    database_context_requirements
    .groupby(
        "required_before_reconstruction",
        as_index=False,
    )
    .agg(
        requirement_count=("requirement", "size"),
        requirements=("requirement", lambda values: " | ".join(values)),
    )
)

reconstruction_readiness["status"] = reconstruction_readiness[
    "required_before_reconstruction"
].map(
    {
        True: "must be designed into the staging database",
        False: "may be added later through research enrichment",
    }
)

reconstruction_readiness[
    [
        "required_before_reconstruction",
        "requirement_count",
        "status",
        "requirements",
    ]
]

,required_before_reconstruction,requirement_count,status,requirements
0,False,3,may be added later through research enrichment,Do not treat source type as a universal native code | Support authority and rules records with effective periods | Track unresolved questions without forcing classifications
1,True,3,must be designed into the staging database,Retain source type exactly as supplied | Store reproducible candidate jurisdiction separately | Preserve raw SP and interpret its market meaning separately


## Reconstruction readiness

Notebook 09 confirms that worldwide regulatory and betting-market research is not a prerequisite for rebuilding the core database.

Three requirements must be designed into reconstruction from the beginning:

- preserve the source race type exactly as supplied
- store candidate jurisdiction as a separate reproducible derivation
- preserve raw starting-price values without assuming that they represent the same market concept in every jurisdiction

Detailed native racing-code mappings, regulatory histories and unresolved interpretation questions can remain in later research-enrichment layers.

This means core race and runner reconstruction may proceed once the source-preservation and structural-derivation requirements from Notebooks 01–09 have been consolidated.

Country-specific studies can later add evidence-backed interpretation records without rewriting the original source or reconstructed race records.

## Notebook 09 conclusions

### Conclusion

The source database does not contain a universal racing-code or betting-market model.

Fields such as `type`, carried weight and `sp` must therefore be treated as source values whose meaning may depend on jurisdiction, racing code, effective period and provider convention.

### Evidence

The study established that:

- candidate jurisdiction can be derived reproducibly for all 189,043 provisional races
- Great Britain requires racing-code context beneath one regulatory authority
- Ireland requires both racing-code and historical-period context
- French source-labelled `NH Flat` races expose a provider classification that cannot yet be safely translated into a native racing code
- regulatory and market interpretation must remain separate from source preservation and structural derivation

### Confidence

Confidence is high in the course and candidate-jurisdiction mapping and in the need for a separate research-enrichment layer.

Confidence remains incomplete for country-specific racing-code and betting-market interpretations not fully researched in this notebook.

### Practical implication

Core database reconstruction can proceed without completing worldwide racing research, provided that it:

- preserves original source values
- stores candidate jurisdiction separately
- does not use source `type` as a universal native racing code
- preserves raw `sp` without assigning one universal market meaning
- allows later evidence-backed enrichment by jurisdiction, code and effective period

### Next action

Create Notebook 10 to inventory and triage every remaining source field.

The notebook will identify:

- fields already investigated
- fields that can be preserved without further semantic work
- fields requiring dedicated profiling notebooks
- fields requiring later jurisdiction-specific research

After the remaining field studies are complete, consolidate the reconstruction requirements into a conceptual staging model before defining the physical schema.